In [ ]:
import os
import requests
from urllib.parse import urlparse

GITHUB_TOKEN = os.getenv("")

headers = {}
if GITHUB_TOKEN:
    headers["Authorization"] = f"token {GITHUB_TOKEN}"

OUTPUT_DIR = "architectural_commit_dataset"


def parse_commit_url(commit_url):
    parsed = urlparse(commit_url)
    parts = parsed.path.strip("/").split("/")
    return parts[0], parts[1], parts[3]


def get_commit_data(org, repo, sha):
    url = f"https://api.github.com/repos/{org}/{repo}/commits/{sha}"
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    return r.json()


def download_file(org, repo, file_path, ref):
    raw_url = f"https://raw.githubusercontent.com/{org}/{repo}/{ref}/{file_path}"
    r = requests.get(raw_url, headers=headers)
    if r.status_code != 200:
        return None
    return r.text


def extract_commit_full_context(commit_url):
    org, repo, sha = parse_commit_url(commit_url)
    commit_data = get_commit_data(org, repo, sha)

    parent_sha = commit_data["parents"][0]["sha"]

    dataset_entry = {
        "repo": f"{org}/{repo}",
        "commit_sha": sha,
        "commit_message": commit_data["commit"]["message"],
        "files": []
    }

    for f in commit_data["files"]:
        file_path = f["filename"]
        status = f["status"]

        before_code = None
        after_code = None

        if status != "added":
            before_code = download_file(org, repo, file_path, parent_sha)

        if status != "removed":
            after_code = download_file(org, repo, file_path, sha)

        dataset_entry["files"].append({
            "path": file_path,
            "status": status,
            "before_code": before_code,
            "after_code": after_code,
            "patch": f.get("patch")
        })

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    import json
    output_path = os.path.join(OUTPUT_DIR, f"{sha}.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(dataset_entry, f, indent=4)

    print(f"Saved structured commit context to {output_path}")


if __name__ == "__main__":
    commit_url = "https://github.com/apache/incubator-seata/commit/1e2812ddc066404739907bc4f9a17130ee368167"
    extract_commit_full_context(commit_url)


Saved structured commit context to architectural_commit_dataset/1e2812ddc066404739907bc4f9a17130ee368167.json
